In [ ]:
# Instructions to re-train : simply drop the files from src/ inside colab and run cells

In [1]:
!pip install python-chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 124.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=a07b526023bb47f388af92b3718bdc6f02d3e76229f943d4570d24b0f71c2403
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [ ]:
# CREATE A DATA FOLDER BEFORE !!! CRUCIAL !

In [2]:
!python preprocessing.py

README.md: 4.41kB [00:00, 16.8MB/s]
data/train-00000-of-00003.parquet: 100% 161M/161M [00:09<00:00, 17.3MB/s]
data/train-00001-of-00003.parquet: 100% 161M/161M [00:08<00:00, 19.7MB/s] 
data/train-00002-of-00003.parquet: 100% 161M/161M [00:08<00:00, 19.2MB/s] 
Generating train split: 100% 5682618/5682618 [00:06<00:00, 828321.48 examples/s] 
Tokenizing all FENs...
100% 5682618/5682618 [26:36<00:00, 3559.28it/s]
Done ! Saved encoded FENs to data/encoded_fens.npy


In [3]:
import torch
from torch.utils.data import DataLoader
import json
import os
import time
from tqdm import tqdm
import chess

from data import ChessDataset
from model import AutoRegressiveTransformer
from tokenizer import FENTokenizer


def extract_board_position(fen_str):
    parts = fen_str.split(" ")
    return " ".join(parts[:4])


def build_board_position_hash_set(dataset, tokenizer):
    board_positions = set()
    total = len(dataset)

    for i in tqdm(range(total), desc="Building board position hash set"):
        encoded_fen = dataset.data[i]
        fen_str = tokenizer.decode(encoded_fen.tolist())
        board_position = extract_board_position(fen_str)
        board_positions.add(board_position)

    print(f"Built hash set with {len(board_positions)} unique positions")
    return board_positions


def is_valid_fen(fen_str):
    """Check if a FEN string represents a valid chess position."""
    try:
        chess.Board(fen_str)
        return True
    except:
        return False


def calculate_novelty(generated_fens, board_position_set):
    novel_count = sum(1 for fen in generated_fens
                     if extract_board_position(fen) not in board_position_set)
    return (novel_count / len(generated_fens)) * 100 if generated_fens else 0


EPOCH = 1
BATCH_SIZE = 128

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device {device}")

dataset = ChessDataset("data/encoded_fens.npy")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

tokenizer = FENTokenizer()
board_position_set = build_board_position_hash_set(dataset, tokenizer)
os.makedirs("outputs", exist_ok=True)
loss_history = []
validity_history = []
uniqueness_history = []
novelty_history = []

total_steps = 1500

model = AutoRegressiveTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2*1.5625e-6, weight_decay=2*1.5625e-6)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: (
        1.0 if step < 500 else
        1.0 - (0.75 * (step - 500) / (total_steps - 500))
    )
)

iteration = 0
start_time = time.time()

for epoch in range(EPOCH):
    for batch in dataloader:
        batch = batch.to(device)
        inputs = batch[:, :-1]
        targets = batch[:, 1:]

        logits, loss = model(inputs, targets)
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        if (iteration % 10) == 0:
            loss_val = loss.item()
            loss_history.append((iteration, loss_val))

            model.eval()
            with torch.no_grad():
                start_idx = torch.zeros((100, 1), dtype=torch.long, device=device)
                generated = model.generate(start_idx, max_new_tokens=83, temperature=1.0)

                generated_fens = []
                for i in range(generated.size(0)):
                    token_ids = generated[i].cpu().tolist()
                    try:
                        fen_str = tokenizer.decode(token_ids)
                        generated_fens.append(fen_str)
                    except:
                        continue

                # Filter for valid FENs only
                valid_fens = [fen for fen in generated_fens if is_valid_fen(fen)]

                validity = (len(valid_fens) / 100) * 100
                uniqueness = (len(set(valid_fens)) / len(valid_fens)) * 100 if valid_fens else 0
                novelty = calculate_novelty(valid_fens, board_position_set)
                print(list(set(generated_fens))[:3])

                validity_history.append((iteration, validity))
                uniqueness_history.append((iteration, uniqueness))
                novelty_history.append((iteration, novelty))

            model.train()

            time_taken = time.time() - start_time

            print(f"Iteration {iteration}, Loss: {loss_val:.4f}, "
                  f"Validity: {validity:.1f}%, Uniqueness: {uniqueness:.1f}%, "
                  f"Novelty: {novelty:.1f}%, Time: {time_taken:.2f}s")

            start_time = time.time()

        iteration += 1

Using device cuda


Building board position hash set: 100%|██████████| 5682618/5682618 [01:34<00:00, 59918.20it/s]


Built hash set with 5647034 unique positions
Model Parameter Count: 134.53M
[]
Iteration 0, Loss: 4.0403, Validity: 0.0%, Uniqueness: 0.0%, Novelty: 0.0%, Time: 30.78s
['rq1P15p78Q1p-1191n/2-2b1e1h2p13h4P1n3/51rd6 . p .. 0 pP', 'r2/4k8/14/9a502a1p1/315 . b .. P p', 'rp01cnp1pn56cppe1c5p-1P418q3/59p7Pp p - .. 1g 1']
Iteration 10, Loss: 1.8662, Validity: 0.0%, Uniqueness: 0.0%, Novelty: 0.0%, Time: 54.49s
['rNdp-p1k11R1R23ppPppppP1/2PpP1/5k1-3/4p1/7PBP4-1 . bB .p Qa 1', 'r1/pp2-c/1/1bpn7/4np3w/1ph1P1n4rk10p5p1//10p . p07 .. Pb 1', 'rp1/p2p145Q2P18223pp12p3ap3c1//5-1k3P3-1 . P .. P /']
Iteration 20, Loss: 1.7364, Validity: 0.0%, Uniqueness: 0.0%, Novelty: 0.0%, Time: 58.38s
['rQ1P15/01pq/1n21p5/p0N7n1p-1r12/281/3g3//2p1- P - .p P- 0', 'r6kP11Q10PqR1P22P-h1p2PQ3-1/2P3k1R1-3b2 . - .- / 1', 'r2ppN3/252P/2850512rR3K2n7/2Q4/PPp40/P1k . - .. / -Q']
Iteration 30, Loss: 1.6460, Validity: 0.0%, Uniqueness: 0.0%, Novelty: 0.0%, Time: 57.44s
['r1hp2ap1pr1p3k3b2P192p2/1r41/40p1K/35/3P6BbRP-49-1 . Bd 

KeyboardInterrupt: 

In [4]:
torch.save(model.state_dict(), "model_checkpoint_1500_iterations_128_bs.pt")
torch.save(optimizer.state_dict(), "optimizer_1500_128.pt")


In [ ]:
# !pkill python

In [5]:
print("Saving training metrics...")
metrics_data = {
    "loss": loss_history,
    "validity": validity_history,
    "uniqueness": uniqueness_history,
    "novelty": novelty_history,
}

output_file = os.path.join("outputs", "training_metrics.json")
with open(output_file, "w") as f:
    json.dump(metrics_data, f, indent=4)

print(f"Saved training metrics to {output_file}")

Saving training metrics...
Saved training metrics to outputs/training_metrics.json


In [ ]:
!ls

data					   optimizer_600_128.pt  __pycache__
data.py					   outputs		 sample_data
model_checkpoint_600_iterations_128_bs.pt  plot_metrics.py	 tokenizer.py
model.py				   preprocessing.py
